<a href="https://colab.research.google.com/github/conord2021-commits/GPT2-Coding-Model/blob/main/Coding_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT2 model from scratch with pretrained weights from Huggingface

In [ ]:
import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
from datasets import load_dataset

In [ ]:
# Load dataset
dataset = load_dataset('flytech/python-codes-25k', split='train')

# Map to plain text — instruction + input + output concatenated
dataset = dataset.map(
    lambda example: {'text': example['instruction'] + ' ' + example['input'] + ' ' + example['output']}
)['text']


https://huggingface.co/datasets/flytech/python-codes-25k

## Dataset

## Tokenisation

In [ ]:
enc = tiktoken.get_encoding("gpt2")

eos_token = enc.decode([enc.eot_token])
combined_text = f" {eos_token} ".join(dataset)
tokens        = enc.encode(combined_text, allowed_special={eos_token})
all_tokens    = torch.tensor(tokens, dtype=torch.long)

split_ratio = 0.9
split_at    = int(len(all_tokens) * split_ratio)
train_tokens = all_tokens[:split_at]
val_tokens   = all_tokens[split_at:]

print(f"Prepared {len(train_tokens):,} training tokens and {len(val_tokens):,} validation tokens.")


## Model Architecture

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()  # batch size, sequence length, embedding dimensionality (n_embd)
        # nh is "number of heads", hs is "head size", C = nh * hs
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)      # Flash Attention
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        # output projection
        y = self.c_proj(y)
        return y

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1          # <-- required by _init_weights

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 1024 # max sequence length
    vocab_size: int = 50257 # 50,000 BPE merges + 256 byte tokens + 1 <|endoftext|>
    n_layer: int = 6 # number of transformer layers
    n_head: int = 8 # number of attention heads
    n_embd: int = 768 # embedding dimension

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h   = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight sharing scheme
        self.transformer.wte.weight = self.lm_head.weight

        # initialise all weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2 * self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, \
            f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"
        pos     = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(idx)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)    # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        # Pre - trained weights
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        config_args = {
            'gpt2': dict(n_layer=12, n_head=12, n_embd=768),   # 124M
            'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),  # 350M
            'gpt2-large': dict(n_layer=36, n_head=20, n_embd=1280),  # 774M
            'gpt2-xl': dict(n_layer=48, n_head=25, n_embd=1600),  # 1558M
        }[model_type]

        config_args['vocab_size'] = 50257
        config_args['block_size'] = 1024

        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = [k for k in sd.keys() if not k.endswith('.attn.bias')]

        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = [k for k in sd_hf.keys() if not k.endswith('.attn.masked_bias')]
        sd_keys_hf = [k for k in sd_keys_hf  if not k.endswith('.attn.bias')]

        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight',
                      'mlp.c_fc.weight',    'mlp.c_proj.weight']

        assert len(sd_keys_hf) == len(sd_keys), \
            f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"

        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])
        return model

    def configure_optimizers(self, weight_decay, learning_rate, device_type):
        param_dict   = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params   = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params,   'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        print(f"num decayed parameter tensors:     {len(decay_params)},   "
              f"with {sum(p.numel() for p in decay_params):,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, "
              f"with {sum(p.numel() for p in nodecay_params):,} parameters")
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == "cuda"
        print(f"using fused AdamW: {use_fused}")
        optimizer = torch.optim.AdamW(
            optim_groups, lr=learning_rate,
            betas=(0.9, 0.95), eps=1e-8, fused=use_fused
        )
        return optimizer


In [ ]:
class DataLoaderLite:

    def __init__(self, tokens: torch.Tensor, B: int, T: int):
        self.tokens = tokens
        self.B = B
        self.T = T
        self.current_position = 0
        print(f"loaded {len(self.tokens):,} tokens")
        print(f"1 epoch = {len(self.tokens) // (B * T):,} batches")

    def next_batch(self):
        B, T = self.B, self.T
        if self.current_position + B * T + 1 > len(self.tokens):
            self.current_position = 0
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        self.current_position += B * T
        return x, y


In [ ]:

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"using device: {device}")

device_type = "cuda" if device.startswith("cuda") else "cpu"

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

if device_type == "cuda":
    torch.set_float32_matmul_precision("high")

B = 8    # batch size, change to 16?
T = 512  # sequence length

total_batch_size = 131072  # 128k tokens
assert total_batch_size % (B * T) == 0, "total_batch_size must be divisible by B*T"
grad_accum_steps = total_batch_size // (B * T)
print(f"total desired batch size: {total_batch_size:,}")
print(f"=> gradient accumulation steps: {grad_accum_steps}")

train_loader = DataLoaderLite(train_tokens, B=B, T=T)
val_loader   = DataLoaderLite(val_tokens,   B=B, T=T)

max_lr = 6e-4
min_lr = max_lr * 0.1
max_steps = 300
warmup_steps = 50

def get_lr(step):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

log_dir        = "/content/nanogpt"
os.makedirs(log_dir, exist_ok=True)
log_file       = os.path.join(log_dir, "log.txt")
checkpoint_path = os.path.join(log_dir, "model_checkpoint.pt")

def save_checkpoint(model, optimizer, step, val_loss, path):
    checkpoint = {
        "model":     model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step":      step,
        "config":    model.config,
        "val_loss":  val_loss,
    }
    torch.save(checkpoint, path)
    print(f"checkpoint saved → {path}  (step {step})")

start_step = 0
if os.path.exists(checkpoint_path):
    print(f"resuming from checkpoint: {checkpoint_path}")
    torch.serialization.add_safe_globals([GPTConfig])
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = GPT(ckpt["config"])
    model.load_state_dict(ckpt["model"])
    model.to(device)
    optimizer = model.configure_optimizers(
        weight_decay=0.1, learning_rate=max_lr, device_type=device_type
    )
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"] + 1
    print(f"resuming from step {start_step}, previous val loss: {ckpt['val_loss']:.4f}")
else:
    print("no checkpoint found — loading pretrained GPT-2 weights")
    model = GPT.from_pretrained("gpt2")
    model.to(device)
    optimizer = model.configure_optimizers(
        weight_decay=0.1, learning_rate=max_lr, device_type=device_type
    )

step = start_step # ensure step is defined even if loop is interrupted early
last_val_loss = float("inf")

try:
    with open(log_file, "a") as log_f:
        for step in range(start_step, max_steps):
            t0        = time.time()
            last_step = (step == max_steps - 1)

            # checkpoint every 50 steps ──
            if step % 50 == 0 or last_step:
                model.eval()
                val_loader.current_position = 0
                with torch.no_grad():
                    val_loss_accum = 0.0
                    for _ in range(10):
                        x, y = val_loader.next_batch()
                        x, y = x.to(device), y.to(device)
                        with torch.autocast(device_type=device_type, dtype=torch.bfloat16,
                                            enabled=(device_type in ("cuda", "cpu"))):
                            _, loss = model(x, y)
                        val_loss_accum += loss.item() / 10
                last_val_loss = val_loss_accum
                print(f"step {step:4d} | val loss: {val_loss_accum:.4f}")
                log_f.write(f"{step} val {val_loss_accum:.4f}\n")
                log_f.flush()
                save_checkpoint(model, optimizer, step, val_loss_accum, checkpoint_path)

            # training step
            model.train()
            optimizer.zero_grad()
            loss_accum = 0.0
            for micro_step in range(grad_accum_steps):
                x, y = train_loader.next_batch()
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16,
                                    enabled=(device_type in ("cuda", "cpu"))):
                    logits, loss = model(x, y)
                loss = loss / grad_accum_steps
                loss_accum += loss.detach()
                loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            lr = get_lr(step)
            for pg in optimizer.param_groups:
                pg["lr"] = lr
            optimizer.step()
            if device_type == "cuda":
                torch.cuda.synchronize()

            dt = time.time() - t0
            tokens_per_sec = (B * T * grad_accum_steps) / dt
            if step % 10 == 0:
                print(f"step {step:4d} | loss: {loss_accum.item():.4f} | lr: {lr:.2e} | tok/sec: {tokens_per_sec:.0f}")
            log_f.write(f"{step} train {loss_accum.item():.6f}\n")

except KeyboardInterrupt:
    print("\nTraining interrupted.")

save_checkpoint(model, optimizer, step, last_val_loss, checkpoint_path)
print(f"checkpoint saved to {checkpoint_path}")


In [ ]:
# Testing Cell, fully self contained

import os
import math
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu   = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer:    int = 12
    n_head:     int = 12
    n_embd:     int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, "NANOGPT_SCALE_INIT"):
                std *= (2 * self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size
        pos     = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(idx)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x      = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# Load Model
CHECKPOINT_PATH = "/content/nanogpt/model_checkpoint (5).pt"

assert os.path.exists(CHECKPOINT_PATH), (
    f"Checkpoint not found at {CHECKPOINT_PATH}.\n"
    "Upload model_checkpoint.pt via the Colab file browser (left panel → upload icon)."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")

torch.serialization.add_safe_globals([GPTConfig])
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

config = ckpt["config"]
model  = GPT(config)
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()

trained_steps = ckpt.get("step", "unknown")
val_loss      = ckpt.get("val_loss", float("nan"))
print(f"loaded checkpoint — step: {trained_steps} | val loss: {val_loss:.4f}")

enc = tiktoken.get_encoding("gpt2")

def generate(prompt: str, max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 50):
    tokens = enc.encode(prompt)
    tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond    = tokens[:, -model.config.block_size:]
            logits, _   = model(idx_cond)
            logits      = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _    = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs    = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            tokens   = torch.cat([tokens, next_tok], dim=1)
            if next_tok.item() == enc.eot_token:
                break

    return enc.decode(tokens[0].tolist())

print(generate("create a for loop from 1 to 5:"))
